In [1]:
%pip -q install altair vl-convert-python pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 20.8 MB/s eta 0:00:00


### Cargo la base de datos limpia de la Entrega 2

In [2]:
import pandas as pd
import numpy as np
import altair as alt
from PIL import Image
from google.colab import files
from pathlib import Path

print('Sube el archivo datos_electorales_database_limpia.csv')
subidos = files.upload()
nombre_archivo = 'datos_electorales_database_limpia.csv'
assert nombre_archivo in subidos, 'Debes subir el CSV con ese nombre.'
df = pd.read_csv(nombre_archivo)
print('Filas y columnas:', df.shape)
display(df.head())

Sube el archivo datos_electorales_database_limpia.csv


Saving datos_electorales_database_limpia.csv to datos_electorales_database_limpia.csv
Filas y columnas: (2515, 19)


,id_candidato,nombre_candidato,sexo,region,circunscripcion,letra_lista,pacto,subpacto,sigla_partido,partido,numero_papeleta,posicion_lista,primero_lista,tamano_lista,numero_listas_circunscripcion,votos,votos_lista,porcentaje_votos_lista,electo
0,FERNANDO_ALEX_FERNANDEZ_BARRAZA__ARICA,FERNANDO ALEX FERNANDEZ BARRAZA,H,REGIÓN DE ARICA Y PARINACOTA,ARICA,B,IZQUIERDA ECOLOGISTA POPULAR,NaN,POPULAR,POPULAR,100,1,Sí,11,13,450,7671,5.866250,No
1,SERGIO_LEANDRO_MORALES_IMANA__ARICA,SERGIO LEANDRO MORALES IMAÑA,H,REGIÓN DE ARICA Y PARINACOTA,ARICA,B,IZQUIERDA ECOLOGISTA POPULAR,NaN,PH,PARTIDO HUMANISTA,101,2,No,11,13,665,7671,8.669013,No
2,ANTONIO_LUCIO_MAITA_FLORES__ARICA,ANTONIO LUCIO MAITA FLORES,H,REGIÓN DE ARICA Y PARINACOTA,ARICA,B,IZQUIERDA ECOLOGISTA POPULAR,NaN,PH,PARTIDO HUMANISTA,102,3,No,11,13,281,7671,3.663147,No
3,DAVID_JORGE_ANTONIO_PADILLA_RAMIREZ__ARICA,DAVID JORGE ANTONIO PADILLA RAMIREZ,H,REGIÓN DE ARICA Y PARINACOTA,ARICA,B,IZQUIERDA ECOLOGISTA POPULAR,NaN,PH,PARTIDO HUMANISTA,103,4,No,11,13,251,7671,3.272064,No
4,ALEXANDRA_CASTILLO_BACIAN__ARICA,ALEXANDRA CASTILLO BACIAN,M,REGIÓN DE ARICA Y PARINACOTA,ARICA,B,IZQUIERDA ECOLOGISTA POPULAR,NaN,PH,PARTIDO HUMANISTA,104,5,No,11,13,828,7671,10.793899,No


#### Reviso que la base se haya cargado correctamente y compruebo las variables que necesito para hacer el gráfico

In [3]:
columnas = ['region', 'electo', 'posicion_lista', 'primero_lista']
assert all(col in df.columns for col in columnas)
assert df[columnas].notna().all().all(), 'Revisar datos faltantes en columnas necesarias.'
assert df['posicion_lista'].ge(1).all()
assert ((df['posicion_lista'] == 1) == (df['primero_lista'] == 'Sí')).all()
assert df['electo'].isin(['Sí', 'No']).all()

electos = df.loc[df['electo'] == 'Sí'].copy()
electos['categoria'] = np.where(electos['posicion_lista'] == 1, 'Primera posición', 'Otras posiciones')
print('Total de candidaturas:', len(df))
print('Candidaturas electas:', len(electos))
print(electos['categoria'].value_counts())
assert len(df) == 2515 and len(electos) == 302
assert (electos['categoria'] == 'Primera posición').sum() == 204

Total de candidaturas: 2515
Candidaturas electas: 302
categoria
Primera posición    204
Otras posiciones     98
Name: count, dtype: int64


Para esta visualización solo usaré las candidaturas que resultaron electas. Las separo en dos grupos:
- quienes estaban en la primera posición de su lista
- quienes ocupaban cualquier otra posición.

Veo cuántas candidaturas electas hay en cada grupo y región. Luego calculo qué porcentaje representa cada grupo del total de electos de su región.

In [19]:
### Defino el orden de las regiones de norte a sur
resumen = (electos.groupby(['region', 'categoria']).size()
           .unstack(fill_value=0).reset_index())
for categoria in ['Primera posición', 'Otras posiciones']:
    if categoria not in resumen.columns:
        resumen[categoria] = 0
resumen['total'] = resumen['Primera posición'] + resumen['Otras posiciones']
resumen['porcentaje_primera'] = 100 * resumen['Primera posición'] / resumen['total']

nombres = {'REGIÓN DE ARICA Y PARINACOTA': 'Arica y Parinacota', 'REGIÓN DE TARAPACÁ': 'Tarapacá', 'REGIÓN DE ANTOFAGASTA': 'Antofagasta', 'REGIÓN DE ATACAMA': 'Atacama', 'REGIÓN DE COQUIMBO': 'Coquimbo', 'REGIÓN DE VALPARAÍSO': 'Valparaíso', 'REGIÓN METROPOLITANA DE SANTIAGO': 'Metropolitana', "REGIÓN DEL LIBERTADOR GENERAL BERNARDO O'HIGGINS": "O'Higgins", 'REGIÓN DEL MAULE': 'Maule', 'REGIÓN DE ÑUBLE': 'Ñuble', 'REGIÓN DEL BIOBÍO': 'Biobío', 'REGIÓN DE LA ARAUCANÍA': 'La Araucanía', 'REGIÓN DE LOS RÍOS': 'Los Ríos', 'REGIÓN DE LOS LAGOS': 'Los Lagos', 'REGIÓN DE AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO': 'Aysén', 'REGIÓN DE MAGALLANES Y DE LA ANTÁRTICA CHILENA': 'Magallanes'}
resumen['region_corta'] = resumen['region'].map(nombres)
assert resumen['region_corta'].notna().all()
resumen = resumen.sort_values('porcentaje_primera', ascending=False)

datos_grafico = resumen.melt(
    id_vars=['region', 'region_corta', 'total', 'porcentaje_primera'],
    value_vars=['Primera posición', 'Otras posiciones'],
    var_name='categoria', value_name='cantidad'
)
datos_grafico['porcentaje'] = 100 * datos_grafico['cantidad'] / datos_grafico['total']
datos_grafico['orden_segmento'] = datos_grafico['categoria'].map({'Primera posición': 0, 'Otras posiciones': 1})
datos_grafico.to_csv('datos_para_visualizacion.csv', index=False)
assert len(resumen) == 16 and len(datos_grafico) == 32
assert np.allclose(datos_grafico.groupby('region_corta')['porcentaje'].sum(), 100)
display(resumen[['region_corta', 'Primera posición', 'Otras posiciones', 'total', 'porcentaje_primera']])

categoria,region_corta,Primera posición,Otras posiciones,total,porcentaje_primera
15,Metropolitana,30,4,34,88.235294
0,Antofagasta,14,2,16,87.500000
8,Magallanes,12,2,14,85.714286
10,Valparaíso,21,7,28,75.000000
12,Biobío,20,8,28,71.428571
3,Aysén,10,4,14,71.428571
5,La Araucanía,14,6,20,70.000000
14,Maule,14,6,20,70.000000
4,Coquimbo,11,5,16,68.750000
13,O'Higgins,12,8,20,60.000000


In [18]:
orden_regiones = [
    "Arica y Parinacota", "Tarapacá", "Antofagasta",
    "Atacama", "Coquimbo", "Valparaíso",
    "Metropolitana", "O'Higgins", "Maule",
    "Ñuble", "Biobío", "La Araucanía",
    "Los Ríos", "Los Lagos", "Aysén", "Magallanes"
]

colores = ["#246A8F", "#E4A76C"]

### Creo un gráfico de barras horizontales con Altair.
### Cada barra representa una región y los colores permiten distinguir a los electos que estaban en primera posición de los que ocupaban otras posiciones.

base = alt.Chart(datos_grafico).encode(
    y=alt.Y(
        "region_corta:N",
        sort=orden_regiones,
        title=None
    ),
    x=alt.X(
        "sum(porcentaje):Q",
        stack="zero",
        scale=alt.Scale(domain=[0, 100]),
        axis=alt.Axis(
            title="Porcentaje de candidaturas electas",
            values=[0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100],
            labelExpr="datum.value + '%'"
        )
    ),
    color=alt.Color(
        "categoria:N",
        scale=alt.Scale(
            domain=["Primera posición", "Otras posiciones"],
            range=colores
        ),
        legend=alt.Legend(title="Posición dentro de la lista")
    ),
    order=alt.Order("orden_segmento:Q", sort="ascending"),
    tooltip=[
        alt.Tooltip("region_corta:N", title="Región"),
        alt.Tooltip("categoria:N", title="Posición"),
        alt.Tooltip("cantidad:Q", title="Electos"),
        alt.Tooltip("total:Q", title="Total electos en la región"),
        alt.Tooltip("porcentaje:Q", title="Porcentaje", format=".1f")
    ]
)

barras = base.mark_bar()

grafico = barras.properties(    width=650,
    height=alt.Step(30),
    title=alt.TitleParams(
        text="¿Qué posición ocupaban las candidaturas electas dentro de su lista?",
        subtitle="Elecciones CORE 2024 · Composición porcentual por región"
    )
).configure_view(
    stroke=None
).configure_axis(
    labelFontSize=12,
    titleFontSize=12
).configure_legend(
    labelFontSize=12
)

grafico

alt.Chart(...)

In [24]:
# Guardo el gráfico interactivo en HTML.
grafico.save("vis_01.html")

# Guardo el gráfico como imagen.
grafico.save("vis_01.png")

In [26]:
from google.colab import files

# Descargo el gráfico interactivo.
files.download("vis_01.html")

# Descargo la imagen del gráfico.
files.download("vis_01.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>